**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后作业文件夹的路径，
# 例如 'cs231n/assignments/assignment1/'。
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 挂载 Drive 后，下面这行让 Colab VM 的 Python 解释器
# 能够加载该作业目录中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 如果 CIFAR-10 数据集还不存在，则将其下载到你的 Drive。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# k-Nearest Neighbor（kNN）练习

*请完成并提交这份 worksheet，包括其中的输出以及 worksheet 外部的任何辅助代码。更多细节请参考课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

kNN 分类器包含两个阶段：

- 训练阶段：分类器接收训练数据，并简单地记住它们。
- 测试阶段：kNN 将每张测试图像与所有训练图像进行比较，并把最相似的 k 个训练样本的标签转移过来进行分类。
- k 的取值通过交叉验证确定。

在这个练习中，你将实现这些步骤，并理解图像分类的基本流程、交叉验证，同时熟练编写高效的向量化代码。

In [ ]:
# 运行本 notebook 的初始化代码。

import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# 这是一点 IPython magic，让 matplotlib 图像内嵌显示在 notebook 中
# 而不是显示在新窗口中。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# 另一点 magic，用于让 notebook 重新加载外部 Python 模块；
# 参考 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [ ]:
# 加载原始 CIFAR-10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# 清理变量，避免多次加载数据导致内存问题
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# 作为合理性检查，打印训练数据和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# 可视化数据集中的一些样本。
# 这里展示每个类别中的若干训练图像。
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# 为了让本练习中的代码运行更快，这里对数据做子采样。
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

# 将图像数据 reshape 成行向量。
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
print(X_train.shape, X_test.shape)

In [ ]:
from cs231n.classifiers import KNearestNeighbor

# 创建一个 kNN 分类器实例。
# 记住：训练 kNN 分类器本质上什么也不计算；
# 分类器只是记住训练数据，不做进一步处理。
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)

现在我们希望使用 kNN 分类器对测试数据进行分类。回忆一下，这个过程可以分成两步：

1. 首先计算所有测试样本和所有训练样本之间的距离。
2. 给定这些距离后，对每个测试样本找到 k 个最近的训练样本，并让它们对标签投票。

先从计算所有训练样本和测试样本之间的距离矩阵开始。例如，如果有 **Ntr** 个训练样本和 **Nte** 个测试样本，这一步应该得到一个 **Nte x Ntr** 的矩阵，其中元素 `(i, j)` 表示第 i 个测试样本和第 j 个训练样本之间的距离。

**注意：本 notebook 要求你实现三种距离计算方式，在这些实现中不能使用 numpy 提供的 `np.linalg.norm()` 函数。**

首先打开 `cs231n/classifiers/k_nearest_neighbor.py`，实现 `compute_distances_two_loops` 函数。这个函数使用一个非常低效的双重循环，遍历所有（测试样本、训练样本）对，并逐元素计算距离矩阵。

In [ ]:
# 打开 cs231n/classifiers/k_nearest_neighbor.py，
# 并实现 compute_distances_two_loops。

# 测试你的实现：
dists = classifier.compute_distances_two_loops(X_test)
print(dists.shape)

In [ ]:
# 可以可视化距离矩阵：每一行对应一个测试样本，
# 该行中的数值表示它到各个训练样本的距离。
plt.imshow(dists, interpolation='none')
plt.show()

**内联问题 1**

注意距离矩阵中的结构化模式：有些行或列明显更亮。（默认配色中，黑色表示距离小，白色表示距离大。）

- 数据中的什么原因会导致明显更亮的行？
- 什么原因会导致更亮的列？

$\color{blue}{\textit 你的回答：}$ *在此填写。*

In [ ]:
# 现在实现 predict_labels 函数，并运行下面的代码：
# 这里使用 k = 1，也就是最近邻分类器。
y_test_pred = classifier.predict_labels(dists, k=1)

# 计算并打印预测正确的样本比例。
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

你应该看到大约 `27%` 的准确率。现在尝试更大的 `k`，例如 `k = 5`：

In [ ]:
y_test_pred = classifier.predict_labels(dists, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

你应该会看到它比 `k = 1` 的表现略好。

**内联问题 2**

我们也可以使用其他距离度量，例如 L1 距离。对于某张图像 $I_k$ 在位置 $(i,j)$ 的像素值 $p_{ij}^{(k)}$，

所有图像所有像素上的均值 $\mu$ 为：
$$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$
所有图像在每个像素位置上的逐像素均值 $\mu_{ij}$ 为：
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$
整体标准差 $\sigma$ 和逐像素标准差 $\sigma_{ij}$ 也以类似方式定义。

如果最近邻分类器使用 L1 距离，下列哪些预处理步骤不会改变它的性能？请选择所有适用项。为避免歧义，训练样本和测试样本都会以相同方式预处理。

1. 减去均值 $\mu$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$）。
2. 减去逐像素均值 $\mu_{ij}$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$）。
3. 减去均值 $\mu$，再除以标准差 $\sigma$。
4. 减去逐像素均值 $\mu_{ij}$，再除以逐像素标准差 $\sigma_{ij}$。
5. 旋转数据的坐标轴，也就是把所有图像旋转相同角度。旋转造成的空白区域用相同像素值填充，不做插值。

$\color{blue}{\textit 你的回答：}$


$\color{blue}{\textit 你的解释：}$

In [ ]:
# 现在通过部分向量化来加速距离矩阵计算，版本中只保留一个循环。
# 实现 compute_distances_one_loop 函数，然后运行下面的代码：
dists_one = classifier.compute_distances_one_loop(X_test)

# 为了确认向量化实现是正确的，我们检查它是否与朴素实现一致。
# 判断两个矩阵是否相似有很多方法，其中最简单的方法之一是 Frobenius 范数。
# 如果你以前没有见过：两个矩阵差值的 Frobenius 范数，等于所有元素差值平方和的平方根；
# 换句话说，就是把两个矩阵 reshape 成向量后，计算它们之间的欧氏距离。
difference = np.linalg.norm(dists - dists_one, ord='fro')
print('One loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# 现在在 compute_distances_no_loops 中实现完全向量化版本，
# 然后运行下面的代码。
dists_two = classifier.compute_distances_no_loops(X_test)

# 检查该距离矩阵是否与之前计算出的距离矩阵一致：
difference = np.linalg.norm(dists - dists_two, ord='fro')
print('No loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# 比较不同实现的运行速度。
def time_function(f, *args):
    """
    使用 args 调用函数 f，并返回执行耗时（秒）。
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Two loop version took %f seconds' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('One loop version took %f seconds' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('No loop version took %f seconds' % no_loop_time)

# 你应该会看到完全向量化实现的速度明显更快。

# 注意：不同机器上的表现可能不同。
# 从双循环改为单循环时，你可能看不到加速，甚至可能变慢。

### 交叉验证

我们已经实现了 k-Nearest Neighbor 分类器，但之前随意设置了 `k = 5`。现在我们将使用交叉验证来确定这个超参数的最佳取值。

In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
################################################################################
# TODO:                                                                        #
# 将训练数据划分为若干 folds。划分后，X_train_folds 和 y_train_folds            #
# 都应该是长度为 num_folds 的列表，其中 y_train_folds[i] 是                   #
# X_train_folds[i] 中各个样本对应的标签向量。                                  #
# 提示：查阅 numpy.array_split 函数。                                          #
################################################################################


# 这个字典用于保存交叉验证时不同 k 值对应的准确率。
# 交叉验证完成后，k_to_accuracies[k] 应该是一个长度为 num_folds 的列表，
# 存放该 k 值在不同 fold 上得到的准确率。
k_to_accuracies = {}


################################################################################
# TODO:                                                                        #
# 执行 k-fold 交叉验证，找出最佳 k 值。对于每个候选 k，运行 kNN 算法           #
# num_folds 次：每次用其中 num_folds - 1 个 fold 作为训练数据，               #
# 剩下的一个 fold 作为验证集。将所有 fold、所有 k 值的准确率存入              #
# k_to_accuracies 字典。                                                       #
################################################################################


# 打印计算得到的准确率。
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, accuracy = %f' % (k, accuracy))

In [ ]:
# 绘制原始观测点。
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# 绘制趋势线；误差棒对应标准差。
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Cross-validation on k')
plt.xlabel('k')
plt.ylabel('Cross-validation accuracy')
plt.show()

In [ ]:
# 根据上面的交叉验证结果选择最佳 k 值，
# 然后用全部训练数据重新训练分类器，并在测试数据上评估。
# 你应该能在测试数据上得到超过 28% 的准确率。
best_k = 1

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# 计算并显示准确率。
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

**内联问题 3**

在分类设置中，关于 $k$-Nearest Neighbor（$k$-NN）的下列说法哪些对所有 $k$ 都成立？请选择所有适用项。
1. k-NN 分类器的决策边界是线性的。
2. 1-NN 的训练误差总是小于或等于 5-NN 的训练误差。
3. 1-NN 的测试误差总是小于 5-NN 的测试误差。
4. 使用 k-NN 分类器对一个测试样本分类所需的时间，会随着训练集大小增长而增长。
5. 以上都不对。

$\color{blue}{\textit 你的回答：}$


$\color{blue}{\textit 你的解释：}$